In [1]:
import pynapple as nap
import numpy as np
import pandas as pd
from pathlib import Path

path = Path('/Users/iii9781/pynaviz/playground')

In [2]:
# Remapping functions
def get_shank_id(x):
    """Assign shank ID (1-4) based on x position."""
    if x < 125: return 1
    elif x < 375: return 2
    elif x < 600: return 3
    else: return 4

def extract_cluster_metadata(positions, clusters, good_ids):
    """Extract median x,y positions for good clusters."""
    unique_clusters = np.unique(clusters)
    cluster_positions = {c: np.median(positions[clusters == c], axis=0) for c in unique_clusters}
    df = pd.DataFrame.from_dict(cluster_positions, orient='index', columns=['x', 'y'])
    df.index.name = 'cluster_id'
    return df[df.index.isin(good_ids)]

def create_new_cluster_ids(cluster_pos_df):
    """Create new IDs: rank_by_depth + 50*(shank_id-1)."""
    df = cluster_pos_df.copy()
    df['shank_id'] = df['x'].apply(get_shank_id)
    df['rank_in_shank'] = df.groupby('shank_id')['y'].rank(method='dense').astype(int)
    df['new_cluster_id'] = df['rank_in_shank'] + 50 * (df['shank_id'] - 1)
    return df, df['new_cluster_id'].to_dict()

def build_tsgroup(spikes, clusters, good_ids, id_mapping, metadata_df):
    """Build TsGroup with remapped cluster IDs."""
    spikes_group = {id_mapping[c]: spikes[clusters == c] for c in good_ids}
    new_meta = metadata_df.set_index('new_cluster_id')[['x', 'y', 'shank_id']]
    new_meta.index.name = 'cluster_id'
    return nap.TsGroup(spikes_group, metadata=new_meta.loc[list(spikes_group.keys())])

## Probe A

In [12]:
# Step 1: Load raw data
probe_path_A = path / 'prbA'
posA = np.load('/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Yiwen/mouse3/20250919_hunt_sleep/kilosort4_rigYJ_probeA/spike_positions.npy', mmap_mode='r')
spikesA = np.load('/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Tuguldur/mouse_hunt_sleep/prbA/adc_spike_times.npy', mmap_mode='r')
clustersA = np.load(probe_path_A / 'spike_clusters.npy', mmap_mode='r')
clu_infoA = pd.read_csv(probe_path_A / 'cluster_group.tsv', sep='\t', index_col=0)
print(f"Spike positions shape: {posA.shape}")
print(f"Spike timestamps: {len(spikesA)}")
print(f"Cluster assignments: {len(clustersA)}")
clu_infoA

Spike positions shape: (21379288, 2)
Spike timestamps: 21379288
Cluster assignments: 21379288


,group
cluster_id,
0,noise
1,noise
2,noise
3,good
4,noise
...,...
684,good
685,noise
686,good


In [13]:
# Step 2: Filter for good units only
good_idsA = clu_infoA.index[clu_infoA['group'] == 'good']
print(f"Good clusters: {len(good_idsA)} out of {len(clu_infoA)}")
good_idsA.values

Good clusters: 146 out of 681


array([  3,  10,  12,  16,  17,  21,  22,  43,  44,  50,  52,  53,  54,
        55,  56,  58,  59,  60,  61,  62,  65,  66,  67,  70,  71,  73,
        74,  77,  78,  81,  99, 105, 108, 112, 114, 118, 119, 121, 125,
       129, 132, 135, 139, 142, 144, 147, 156, 161, 171, 176, 178, 181,
       183, 185, 187, 192, 199, 202, 204, 208, 210, 212, 228, 261, 278,
       279, 280, 287, 289, 297, 304, 305, 314, 319, 320, 328, 329, 338,
       339, 354, 369, 374, 375, 387, 388, 399, 407, 409, 420, 432, 435,
       438, 449, 490, 499, 500, 501, 503, 509, 511, 513, 514, 517, 527,
       529, 537, 544, 549, 554, 557, 558, 562, 563, 572, 573, 578, 584,
       606, 607, 619, 622, 625, 627, 631, 633, 635, 638, 639, 640, 641,
       644, 647, 649, 659, 667, 668, 670, 671, 672, 675, 679, 682, 684,
       686, 691, 692])

In [14]:
# Step 3: Compute median x,y position for each cluster
unique_clusters = np.unique(clustersA)
cluster_positions = {c: np.median(posA[clustersA == c], axis=0) for c in unique_clusters}

cluA = pd.DataFrame.from_dict(cluster_positions, orient='index', columns=['x', 'y'])
cluA.index.name = 'cluster_id'
cluA = cluA[cluA.index.isin(good_idsA)]  # Keep only good clusters
print(f"Cluster positions for {len(cluA)} good units")
cluA

Cluster positions for 146 good units


,x,y
cluster_id,,
3,20.0,958.253784
10,20.0,1069.566772
12,20.0,1089.224365
16,20.0,1117.724731
17,20.0,1161.182129
...,...,...
682,500.0,999.692139
684,750.0,1129.627686
686,770.0,2197.146973


In [15]:
# Step 4: Assign shank ID based on x position
# Shank 1: x < 125, Shank 2: 125-375, Shank 3: 375-600, Shank 4: x > 600
cluA['shank_id'] = cluA['x'].apply(get_shank_id)
print("Units per shank:")
print(cluA['shank_id'].value_counts().sort_index())
cluA

Units per shank:
shank_id
1    30
2    33
3    33
4    50
Name: count, dtype: int64


,x,y,shank_id
cluster_id,,,
3,20.0,958.253784,1
10,20.0,1069.566772,1
12,20.0,1089.224365,1
16,20.0,1117.724731,1
17,20.0,1161.182129,1
...,...,...,...
682,500.0,999.692139,3
684,750.0,1129.627686,4
686,770.0,2197.146973,4


In [16]:
# Step 5: Rank clusters by depth (y) within each shank
cluA['rank_in_shank'] = cluA.groupby('shank_id')['y'].rank(method='dense').astype(int)
cluA.sort_values(['shank_id', 'rank_in_shank'])

,x,y,shank_id,rank_in_shank
cluster_id,,,,
3,20.0,958.253784,1,1
10,20.0,1069.566772,1,2
12,20.0,1089.224365,1,3
16,20.0,1117.724731,1,4
17,20.0,1161.182129,1,5
...,...,...,...,...
668,770.0,2240.621094,4,46
670,770.0,2243.135254,4,47
672,770.0,2260.077881,4,48


In [17]:
# Step 6: Create new cluster IDs = rank + 50*(shank_id - 1)
# Shank 1: 1-50, Shank 2: 51-100, Shank 3: 101-150, Shank 4: 151-200
cluA['new_cluster_id'] = cluA['rank_in_shank'] + 50 * (cluA['shank_id'] - 1)
mappingA = cluA['new_cluster_id'].to_dict()

print("Old ID -> New ID mapping:")
cluA[['x', 'y', 'shank_id', 'rank_in_shank', 'new_cluster_id']].sort_values(['shank_id', 'new_cluster_id'])

Old ID -> New ID mapping:


,x,y,shank_id,rank_in_shank,new_cluster_id
cluster_id,,,,,
3,20.0,958.253784,1,1,1
10,20.0,1069.566772,1,2,2
12,20.0,1089.224365,1,3,3
16,20.0,1117.724731,1,4,4
17,20.0,1161.182129,1,5,5
...,...,...,...,...,...
668,770.0,2240.621094,4,46,196
670,770.0,2243.135254,4,47,197
672,770.0,2260.077881,4,48,198


In [18]:
# Step 7: Build spike dictionary with new cluster IDs
spikes_groupA = {mappingA[c]: spikesA[clustersA == c] for c in good_idsA}
print(f"Spike groups created: {len(spikes_groupA)}")
print(f"New cluster IDs: {sorted(spikes_groupA.keys())}")

Spike groups created: 146
New cluster IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200]


In [19]:
# Step 8: Create TsGroup with remapped IDs and metadata
new_metaA = cluA.set_index('new_cluster_id')[['x', 'y', 'shank_id']]
new_metaA.index.name = 'cluster_id'

tsgA = nap.TsGroup(spikes_groupA, metadata=new_metaA.loc[sorted(spikes_groupA.keys())])
tsgA

/var/folders/6s/pwpy5k9j0bq1dj7n4169wcj80000gp/T/ipykernel_28143/3580361494.py:5: UserWarning: Elements should not be passed as <class 'numpy.ndarray'>. Default time units is seconds when creating the Ts object.
  tsgA = nap.TsGroup(spikes_groupA, metadata=new_metaA.loc[sorted(spikes_groupA.keys())])


Index    rate      x      y        shank_id
-------  --------  -----  -------  ----------
1        9.61859   20.0   958.25   1
2        8.2805    20.0   1069.57  1
3        4.13288   20.0   1089.22  1
4        25.19847  20.0   1117.72  1
5        3.03504   20.0   1161.18  1
6        5.20644   20.0   1179.7   1
7        12.95429  20.0   1222.07  1
...      ...       ...    ...      ...
194      7.145     770.0  2060.12  4
195      4.47239   770.0  2126.49  4
196      5.0835    770.0  2080.63  4
197      0.16085   770.0  2122.9   4
198      11.1143   770.0  2197.15  4
199      10.86638  770.0  2199.3   4
200      2.42043   770.0  2214.81  4

## Probe B

In [21]:
# Load probe B data
probe_path_B = path / 'prbB'
posB = np.load('/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Yiwen/mouse3/20250919_hunt_sleep/kilosort4_rigYJ_probeB/spike_positions.npy', mmap_mode='r')
spikesB = np.load('/Volumes/fsmresfiles/Basic_Sciences/Phys/SenzaiLab/Tuguldur/mouse_hunt_sleep/prbB/adc_spike_times.npy', mmap_mode='r')
clustersB = np.load(probe_path_B / 'spike_clusters.npy', mmap_mode='r')
clu_infoB = pd.read_csv(probe_path_B / 'cluster_group.tsv', sep='\t', index_col=0)
good_idsB = clu_infoB.index[clu_infoB['KSLabel'] == 'good']  # Different column name

In [22]:
# Extract metadata and create new IDs
cluB = extract_cluster_metadata(posB, clustersB, good_idsB)
cluB_new, mappingB = create_new_cluster_ids(cluB)
cluB_new[['x', 'y', 'shank_id', 'new_cluster_id']].sort_values(['shank_id', 'new_cluster_id'])

,x,y,shank_id,new_cluster_id
cluster_id,,,,
4,20.0,937.255859,1,1
6,20.0,953.327026,1,2
10,20.0,956.719177,1,3
12,20.0,963.980469,1,4
24,20.0,1031.325684,1,5
...,...,...,...,...
605,770.0,2109.821777,4,155
609,770.0,2121.965088,4,156
611,770.0,2189.333008,4,157


In [23]:
# Build remapped TsGroup
tsgB = build_tsgroup(spikesB, clustersB, good_idsB, mappingB, cluB_new)
tsgB

/var/folders/6s/pwpy5k9j0bq1dj7n4169wcj80000gp/T/ipykernel_28143/655316034.py:30: UserWarning: Elements should not be passed as <class 'numpy.ndarray'>. Default time units is seconds when creating the Ts object.
  return nap.TsGroup(spikes_group, metadata=new_meta.loc[list(spikes_group.keys())])


Index    rate      x      y        shank_id
-------  --------  -----  -------  ----------
1        27.36776  20.0   937.26   1
2        0.02948   20.0   953.33   1
3        0.2682    20.0   956.72   1
4        0.07259   20.0   963.98   1
5        0.06986   20.0   1031.33  1
6        0.0305    20.0   1031.8   1
7        2.70584   20.0   1035.31  1
...      ...       ...    ...      ...
153      0.18854   770.0  1952.62  4
154      0.04422   770.0  1985.27  4
155      0.00179   770.0  2109.82  4
156      0.27467   770.0  2121.97  4
157      0.47548   770.0  2189.33  4
158      0.01261   770.0  2215.8   4
159      2.57038   770.0  2281.97  4

In [24]:
# Save remapped TsGroups
tsgA.save('tsgA_remapped.npz')
tsgB.save('tsgB_remapped.npz')